In [ ]:
%env WORKDIR=/tmp/vault
%env VAULT_K8S_NAMESPACE=vaultpr
%env VAULT_HELM_RELEASE_NAME=vaultpr
%env VAULT_SERVICE_NAME=vaultpr-internal 
%env K8S_CLUSTER_NAME=cluster.local

In [ ]:
%%bash
export VAULT_K8S_NAMESPACE="vaultpr" \
export VAULT_HELM_RELEASE_NAME="vaultpr" \
export VAULT_SERVICE_NAME="vaultpr-internal" \
export K8S_CLUSTER_NAME="cluster.local" \
export WORKDIR=/tmp/vault


cat > ${WORKDIR}/vaultpr-csr.conf <<EOF
[req]
default_bits = 2048
prompt = no
encrypt_key = yes
default_md = sha256
distinguished_name = kubelet_serving
req_extensions = v3_req
[ kubelet_serving ]
O = system:nodes
CN = system:node:*.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
[ v3_req ]
basicConstraints = CA:FALSE
keyUsage = nonRepudiation, digitalSignature, keyEncipherment, dataEncipherment
extendedKeyUsage = serverAuth, clientAuth
subjectAltName = @alt_names
[alt_names]
DNS.1 = *.${VAULT_SERVICE_NAME}
DNS.2 = *.${VAULT_SERVICE_NAME}.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
DNS.3 = *.${VAULT_HELM_RELEASE_NAME}
DNS.4 = *.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
IP.1 = 127.0.0.1
EOF

openssl req -new -key ${WORKDIR}/vault.key -out ${WORKDIR}/vaultpr.csr -config ${WORKDIR}/vaultpr-csr.conf


cat > ${WORKDIR}/csrpr.yaml <<EOF
apiVersion: certificates.k8s.io/v1
kind: CertificateSigningRequest
metadata:
   name: vaultpr.svc
spec:
   signerName: kubernetes.io/kubelet-serving
   expirationSeconds: 8640000
   request: $(cat ${WORKDIR}/vaultpr.csr|base64|tr -d '\n')
   usages:
   - digital signature
   - key encipherment
   - server auth
EOF

kubectl create -f ${WORKDIR}/csrpr.yaml

In [ ]:
%%bash
kubectl certificate approve vaultpr.svc

In [ ]:
%%bash
kubectl get csr vaultpr.svc

In [ ]:
%%bash
kubectl get csr vaultpr.svc -o jsonpath='{.status.certificate}' | openssl base64 -d -A -out ${WORKDIR}/vaultpr.crt

In [ ]:
%%bash


kubectl create namespace $VAULT_K8S_NAMESPACE

kubectl create secret generic vault-ha-tls \
   -n $VAULT_K8S_NAMESPACE \
   --from-file=vault.key=${WORKDIR}/vault.key \
   --from-file=vault.crt=${WORKDIR}/vaultpr.crt \
   --from-file=vault.ca=${WORKDIR}/vault.ca


In [ ]:
%%bash 
secret=$(cat vault.hclic)
kubectl create secret generic vault-ent-license --from-literal="license=${secret}" -n $VAULT_K8S_NAMESPACE

In [ ]:
%%bash
cat > ${WORKDIR}/overrides_pr.yaml <<EOF
global:
   enabled: true
   tlsDisable: false # Disabling TLS to avoid issues when connecting to Vault via port forwarding
injector:
   enabled: false
server:
# config.yaml
   image:
      repository: hashicorp/vault-enterprise
      tag: 2.0.3-ent
   enterpriseLicense:
      secretName: vault-ent-license
   extraEnvironmentVars:
      VAULT_CACERT: /vault/userconfig/vault-ha-tls/vault.ca
      VAULT_TLSCERT: /vault/userconfig/vault-ha-tls/vault.crt
      VAULT_TLSKEY: /vault/userconfig/vault-ha-tls/vault.key
      VAULT_CLIENT_CERT: /vault/userconfig/vault-ha-tls/vault.crt
      VAULT_CLIENT_KEY: /vault/userconfig/vault-ha-tls/vault.key
   volumes:
      - name: userconfig-vault-ha-tls
        secret:
         defaultMode: 420
         secretName: vault-ha-tls
   volumeMounts:
      - mountPath: /vault/userconfig/vault-ha-tls
        name: userconfig-vault-ha-tls
        readOnly: true
   standalone:
      enabled: false
   affinity: ""
   ha:
      enabled: true
      replicas: 3
      raft:
         enabled: true
         setNodeId: true
         config: |
            ui = true
            cluster_name = "vault-perf-secondary"
            listener "tcp" {
               tls_disable = 0 # Disabling TLS to avoid issues when connecting to Vault via port forwarding
               address = "[::]:8200"
               cluster_address = "[::]:8201"
               tls_cert_file = "/vault/userconfig/vault-ha-tls/vault.crt"
               tls_key_file  = "/vault/userconfig/vault-ha-tls/vault.key"
               tls_client_ca_file = "/vault/userconfig/vault-ha-tls/vault.ca"
            }
            storage "raft" {
               path = "/vault/data"
            
               retry_join {
                  auto_join             = "provider=k8s namespace=vaultpr label_selector=\"component=server,app.kubernetes.io/name=vault\""
                  auto_join_scheme      = "https"
                  leader_ca_cert_file   = "/vault/userconfig/vault-ha-tls/vault.ca"
                  leader_tls_servername = "vaultpr-0.vaultpr-internal" #Tiene que matchear una SAN del certificado
               }
            
            }
            disable_mlock = true
            service_registration "kubernetes" {}
EOF

In [ ]:
%%bash
helm install -n $VAULT_K8S_NAMESPACE $VAULT_HELM_RELEASE_NAME hashicorp/vault -f ${WORKDIR}/overrides_pr.yaml

In [ ]:
%%bash
kubectl get events -n vaultpr

In [ ]:
%%bash
kubectl get pods -n vaultpr

In [ ]:
%%bash
kubectl exec -n $VAULT_K8S_NAMESPACE vaultpr-0 -- vault operator init \
    -key-shares=1 \
    -key-threshold=1 \
    -format=json > ${WORKDIR}/clusterpr-keys.json

In [ ]:
%%bash
jq -r ".unseal_keys_b64[]" ${WORKDIR}/clusterpr-keys.json
VAULT_UNSEAL_KEY_PR=$(jq -r ".unseal_keys_b64[]" ${WORKDIR}/clusterpr-keys.json)

In [ ]:
%%bash
kubectl exec -n $VAULT_K8S_NAMESPACE vaultpr-0 -- vault operator unseal $(jq -r ".unseal_keys_b64[]" ${WORKDIR}/clusterpr-keys.json)

In [ ]:
%%bash
kubectl exec -n $VAULT_K8S_NAMESPACE vaultpr-1 -- vault operator unseal $(jq -r ".unseal_keys_b64[]" ${WORKDIR}/clusterpr-keys.json)

In [ ]:
%%bash
kubectl exec -n $VAULT_K8S_NAMESPACE vaultpr-2 -- vault operator unseal $(jq -r ".unseal_keys_b64[]" ${WORKDIR}/clusterpr-keys.json)

In [ ]:
%%bash
kubectl exec -n $VAULT_K8S_NAMESPACE -ti vaultpr-0 -- vault status

In [ ]:
%%bash
kubectl exec -n $VAULT_K8S_NAMESPACE -ti vaultpr-0 --  vault license inspect

In [ ]:
%%bash
cat ${WORKDIR}/clusterpr-keys.json | jq -r ".root_token"

In [ ]:
%%bash
#kubectl -n $VAULT_K8S_NAMESPACE get service vault
kubectl -n vaultpr port-forward service/vaultpr 8300:8200